In [9]:
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

import matplotlib.pyplot as plt

In [10]:
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv").squeeze()

print("Testing data:", X_test.shape)
print("Target data:", y_test.shape)

Testing data: (13358, 21)
Target data: (13358,)


In [11]:
scaler = joblib.load("scaler.pkl")

X_test_scaled = scaler.transform(X_test)

print("Test data scaled successfully.")

Test data scaled successfully.


In [12]:
models = {
    "Logistic Regression": joblib.load("logistic_regression.pkl"),
    "Decision Tree": joblib.load("decision_tree.pkl"),
    "Random Forest": joblib.load("random_forest.pkl"),
    "XGBoost": joblib.load("xgboost.pkl")
}

print("All models loaded successfully.")

All models loaded successfully.


In [14]:
test_results = []

for name, model in models.items():
    model = model["model"]

    if name == "Logistic Regression":
        test_data = X_test_scaled
    else:
        test_data = X_test

    test_pred = model.predict(test_data)

    accuracy = accuracy_score(y_test, test_pred)
    precision = precision_score(y_test, test_pred, zero_division=0)
    recall = recall_score(y_test, test_pred, zero_division=0)
    f1 = f1_score(y_test, test_pred, zero_division=0)

    test_results.append({
        "Model": name,
        "Test Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            test_pred,
            target_names=["No Diabetes", "Diabetes"],
            zero_division=0
        )
    )
    


Logistic Regression
Accuracy : 0.7379
Precision: 0.7267
Recall   : 0.7627
F1 Score : 0.7442

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.75      0.71      0.73      6679
    Diabetes       0.73      0.76      0.74      6679

    accuracy                           0.74     13358
   macro avg       0.74      0.74      0.74     13358
weighted avg       0.74      0.74      0.74     13358


Decision Tree
Accuracy : 0.7262
Precision: 0.6997
Recall   : 0.7925
F1 Score : 0.7432

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.76      0.66      0.71      6679
    Diabetes       0.70      0.79      0.74      6679

    accuracy                           0.73     13358
   macro avg       0.73      0.73      0.72     13358
weighted avg       0.73      0.73      0.72     13358


Random Forest
Accuracy : 0.7440
Precision: 0.7222
Recall   : 0.7932
F1 Score : 0.7560

Classification Report:
       

In [ ]:
test_results_df = pd.DataFrame(test_results)

test_results_df = test_results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

display(
    test_results_df.style.format({
        "Test Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}"
    })
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
for ax, (name, model) in zip(axes.ravel(), models.items()):
    if name == "Logistic Regression":
        test_data = X_test_scaled
    else:
        test_data = X_test
    y_pred = model.predict(test_data)
    cm = confusion_matrix(y_test, y_pred)

    BLUE = "#0B7FAE"
    YELLOW = "#FFD23F"

    cell_colors = [
        [BLUE, YELLOW],
        [YELLOW, BLUE]
    ]
    for i in range(2):
        for j in range(2):

            ax.add_patch(
                plt.Rectangle(
                    (j - 0.5, i - 0.5),
                    1,
                    1,
                    facecolor=cell_colors[i][j],
                    edgecolor="white",
                    linewidth=4
                )
            )

            ax.text(
                j,
                i,
                f"{cm[i, j]}",
                ha="center",
                va="center",
                fontsize=11,
                color="white"
            )

    ax.set_title(name, fontsize=12, pad=15)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        ["No Diabetes", "Diabetes"]
    )
    ax.set_yticks([0, 1])
    ax.set_yticklabels(
        ["No Diabetes", "Diabetes"],
        rotation=90,
        va="center"
    )
    ax.yaxis.tick_right()
    ax.tick_params(
        axis="y",
        labelright=True,
        labelleft=False,
        right=False,
        left=False,
        length=0,
        pad=8
    )
    ax.set_xlabel(
        "Predicted Label",
        labelpad=6
    )

    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_bottom()
    ax.set_ylabel(
        "Actual Label",
        rotation=90,
        labelpad=6
    )
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(1.5, -0.5)
    ax.set_frame_on(False)

plt.tight_layout()
plt.show()